In [ ]:
import sys, os
sys.path.append(os.path.abspath("..")) # Allows us to import from grips and qokit directories
import matplotlib.pyplot as plt, networkx as nx, qokit.maxcut as mc, numpy as np
from grips.QAOA_proxy_interface import QAOA_proxy, QAOA_proxy_expectation
from grips.QAOA_simulator import get_simulator, get_expectation, get_result
from grips.plot_utils import plot_heat_map
from grips.triangle_proxy import TriangleProxy
from grips.paper_proxy import PaperProxy
from grips.normal_proxy import NormalProxy

# The following imports and seval statements make Julia proxy functions available
from juliacall import Main as jl
jl.seval('''
using Pkg
Pkg.activate(joinpath(@__DIR__, "../julia"))
Pkg.instantiate()
using JuliaQAOA
''')

# Define parameter ranges
gammas = np.linspace(0, np.pi, 50)
betas = np.linspace(0, np.pi, 50)
# Probabilities for the Erdos-Renyi graph generation
probabilities = [0.5]
# Whether to use python or julia implementations of the proxies
use_julia = True


def collect_QAOA_parameter_data(G: nx.Graph, gammas: np.ndarray, betas: np.ndarray) -> np.ndarray:
    expectations = np.zeros([len(gammas), len(betas)])
    N = G.number_of_nodes()
    ising_model = mc.get_maxcut_terms(G)
    sim = get_simulator(N, ising_model)

    for i in range(len(gammas)):
        for j in range(len(betas)):
            gamma = np.array([gammas[i]])
            beta = np.array([betas[j]])
            result = get_result(N, ising_model, gamma, beta, sim)
            expectations[i][j] = get_expectation(N, ising_model, gamma, beta, sim, result)

    return expectations


def collect_proxy_parameter_data(proxy, gammas: np.ndarray, betas: np.ndarray) -> np.ndarray:
    expectations = np.zeros([len(gammas), len(betas)])
    for i in range(len(gammas)):
        for j in range(len(betas)):
            gamma = np.array([gammas[i]])
            beta = np.array([betas[j]])
            proxy_amplitudes = QAOA_proxy(proxy, gamma, beta)
            final_proxy_amplitudes = proxy_amplitudes[-1]
            expectation = QAOA_proxy_expectation(proxy, final_proxy_amplitudes)
            expectations[i][j] = expectation

    return expectations


for p in probabilities:
    for N in range(2, 7):
        G = nx.erdos_renyi_graph(N, p, seed=18) # generate graphs
        M = G.number_of_edges()

        # Get the proxies
        if use_julia:
            # Julia Version
            triangle_proxy = jl.TriangleProxy(M, N)
            paper_proxy = jl.PaperProxy(M, N, p)
            normal_proxy = jl.NormalProxy(M, N, 1, 1, 1) # Bad parameters, this proxy is not good like this
        else:
            # Python version
            triangle_proxy = TriangleProxy(M, N) # Will not be good for p != 0.5
            paper_proxy = PaperProxy(M, N, p)
            normal_proxy = NormalProxy(M, N, 1, 1, 1) # Bad parameters, this proxy is not good like this


        # Define data paths
        base_data_path = f"Data/Expectation_Heatmaps/Erdos_Renyi/ER_p={p}"
        os.makedirs(base_data_path, exist_ok=True) # Create directories if they do not exist
        qaoa_data_path = os.path.join(base_data_path, f"QAOA_N={N}_M={M}.npz")
        paper_proxy_data_path = os.path.join(base_data_path, f"PaperProxy_N={N}_M={M}.npz")
        triangle_proxy_data_path = os.path.join(base_data_path, f"TriangleProxy_N={N}_M={M}.npz")
        normal_proxy_data_path = os.path.join(base_data_path, f"NormalProxy_N={N}_M={M}.npz")

        # Define image save paths
        base_img_path = f"Plots/Expectation_Heatmaps/Erdos_Renyi/ER_p={p}"
        os.makedirs(base_img_path, exist_ok=True) # Create directories if they do not exist
        qaoa_img_path = os.path.join(base_img_path, f"QAOA_N={N}_M={M}.png")
        paper_proxy_img_path = os.path.join(base_img_path, f"paper_proxy_N={N}_M={M}.png")
        triangle_proxy_img_path = os.path.join(base_img_path, f"triangle_proxy_N={N}_M={M}.png")
        normal_proxy_img_path = os.path.join(base_img_path, f"normal_proxy_N={N}_M={M}.png")


        # Collect data if it has not already been collected and saved, the Plot
        if not os.path.exists(qaoa_data_path):
            qaoa_expectations = collect_QAOA_parameter_data(G, gammas, betas)
            np.savez(qaoa_data_path, expectations=qaoa_expectations, gammas=gammas, betas=betas)

        # QAOA Plot
        data = np.load(qaoa_data_path)
        qaoa_expectations = data['expectations']
        gammas = data['gammas']
        betas = data['betas']
        _ = plot_heat_map(gammas, betas, qaoa_expectations, f"True QAOA Expectation (N={N},M={M})", "Gamma", "Beta")
        plt.savefig(qaoa_img_path)

        if not os.path.exists(paper_proxy_data_path):
            paper_proxy_expectations = collect_proxy_parameter_data(paper_proxy, gammas, betas)
            np.savez(paper_proxy_data_path, expectations=paper_proxy_expectations, gammas=gammas, betas=betas)

        # Paper Proxy Plot
        data = np.load(paper_proxy_data_path)
        paper_proxy_expectations = data['expectations']
        gammas = data['gammas']
        betas = data['betas']
        _ = plot_heat_map(gammas, betas, paper_proxy_expectations, f"Expectation Proxies from Paper (N={N},M={M})", "Gamma", "Beta")
        plt.savefig(paper_proxy_img_path)
        plt.show()

        if not os.path.exists(triangle_proxy_data_path):
            triangle_proxy_expectations = collect_proxy_parameter_data(triangle_proxy, gammas, betas)
            np.savez(triangle_proxy_data_path, expectations=triangle_proxy_expectations, gammas=gammas, betas=betas)

        # Triangle Proxy Plot
        data = np.load(triangle_proxy_data_path)
        triangle_proxy_expectations = data['expectations']
        gammas = data['gammas']
        betas = data['betas']
        _ = plot_heat_map(gammas, betas, triangle_proxy_expectations, f"Expectation Proxies from Us (Triangle) (N={N},M={M})", "Gamma", "Beta")
        plt.savefig(triangle_proxy_img_path)
        plt.show()

        if not os.path.exists(normal_proxy_data_path):
            normal_proxy_expectations = collect_proxy_parameter_data(normal_proxy, gammas, betas)
            np.savez(normal_proxy_data_path, expectations=normal_proxy_expectations, gammas=gammas, betas=betas)

        # Normal Proxy Plot
        data = np.load(normal_proxy_data_path)
        normal_proxy_expectations = data['expectations']
        gammas = data['gammas']
        betas = data['betas']
        _ = plot_heat_map(gammas, betas, normal_proxy_expectations, f"Expectation Proxies from Us (Normal) (N={N},M={M})", "Gamma", "Beta")
        plt.savefig(normal_proxy_img_path)
        plt.show()